## Step 0 — Gradient Checker

In [106]:
from __future__ import annotations  # lazy annotations: allows tuple[...] / X | None on older kernels

from typing import Callable
import numpy as np

In [107]:
def numeric_gradient(
    scalarFunction: Callable[[np.ndarray], float],
    featureValues: np.ndarray,
    h: float = 1e-5
) -> np.ndarray:
    gradient = np.zeros_like(featureValues, dtype=float)

    for i in range(featureValues.size):
        mask = np.zeros_like(featureValues, dtype=float)
        mask.flat[i] = h
        gradient.flat[i] = (scalarFunction(featureValues + mask) - scalarFunction(featureValues - mask)) / (2 * h)
    return gradient

def stable_softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

### Tests — `numeric_gradient`

In [108]:
def _check(name, got, want, atol=1e-6):
    ok = np.allclose(got, want, atol=atol)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", got)
        print("   want:", want)

# 1. f = sum(x^2)  ->  grad = 2x        (vector input, nonlinear)
x = np.array([3.0, -1.0, 0.5])
_check("sum(x^2) grad == 2x", numeric_gradient(lambda v: np.sum(v**2), x), 2 * x)

# 2. f = sum(c*x)  ->  grad = c         (linear -> constant gradient)
c = np.array([2.0, -3.0, 0.7])
_check("sum(c*x) grad == c", numeric_gradient(lambda v: np.sum(c * v), np.zeros(3)), c)

# 3. matrix input -> gradient keeps the matrix shape
W = np.arange(6, dtype=float).reshape(2, 3)
g = numeric_gradient(lambda M: np.sum(M**2), W)
_check("matrix grad == 2W", g, 2 * W)
_check("matrix grad keeps shape", np.array(g.shape), np.array(W.shape))

# 4. f = sum(sin x) -> grad = cos x     (check vs analytic nonlinear)
x = np.array([0.1, 0.7, -1.2, 2.0])
_check("sum(sin x) grad == cos x", numeric_gradient(lambda v: np.sum(np.sin(v)), x), np.cos(x))

[PASS] sum(x^2) grad == 2x
[PASS] sum(c*x) grad == c
[PASS] matrix grad == 2W
[PASS] matrix grad keeps shape
[PASS] sum(sin x) grad == cos x


### Tests — `stable_softmax`

In [109]:
# reuses _check from the numeric_gradient test cell above
x = np.array([2.0, 1.0, 0.1])
p = stable_softmax(x)

_check("probs sum to 1", p.sum(), 1.0)
_check("all in (0, 1)", np.all((p > 0) & (p < 1)), True)

# matches the naive definition on small, safe inputs
naive = np.exp(x) / np.sum(np.exp(x))
_check("matches naive softmax", p, naive)

# stability + shift-invariance: huge logits stay finite and give the same result
big = stable_softmax(x + 1000)
_check("finite on x + 1000", np.all(np.isfinite(big)), True)
_check("shift-invariant (== p)", big, p)

# monotonic: largest logit keeps the largest probability
_check("argmax preserved", np.argmax(p), np.argmax(x))

[PASS] probs sum to 1
[PASS] all in (0, 1)
[PASS] matches naive softmax
[PASS] finite on x + 1000
[PASS] shift-invariant (== p)
[PASS] argmax preserved


## Step 1 — Linear Layer

In [110]:
class Linear:
    """Fully-connected layer:  Y = X @ W + b

    Shapes:
        X : (batch, n_in)     activations from the previous layer
        W : (n_in, n_out)
        b : (n_out,)
        Y : (batch, n_out)
    """

    def __init__(self, n_in: int, n_out: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        
        # simple init: standard normal weights, zero bias
        self.W: np.ndarray = rng.standard_normal((n_in, n_out))
        self.b: np.ndarray = np.zeros(n_out)
            
        # caches / gradient buffers (filled during forward/backward)
        self.X: np.ndarray | None = None   # previous layer's activations, saved for backward
        self.dW: np.ndarray | None = None
        self.db: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (batch, n_in) -> Y: (batch, n_out)."""
        self.X = X                     # cache X: backward needs it to compute dW
        return X @ self.W + self.b     # b (n_out,) broadcasts across every row

    def backward(self, dY: np.ndarray) -> np.ndarray:
        """dY: (batch, n_out) upstream gradient dL/dY. Returns dX: (batch, n_in)."""
        
        # dW: chain rule + sum over the batch -> X.T @ dY.
        #   local derivative dY/dW is X; a weight is reused across all samples,
        #   so the matmul sums those per-sample contributions. Shape (n_in, n_out).
        self.dW = self.X.T @ dY

        # db: local derivative dY/db is 1, so just sum dY over the batch axis.
        #   Shape (n_out,) -- one gradient per bias.
        self.db = dY.sum(axis=0)

        # dX: gradient to hand back to the previous layer (becomes its dY).
        #   local derivative dY/dX is W, so dX = dY @ W.T. Shape (batch, n_in).
        dX = dY @ self.W.T
        
        return dX

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(W, dW), (b, db)].

        The optimizer reads this each step: values are mutated in place, gradients
        are re-fetched (backward rebinds dW/db to new arrays every call).
        """
        return [(self.W, self.dW), (self.b, self.db)]

    def zero_grad(self) -> None:
        """Reset gradient buffers (called after each optimizer step)."""
        self.dW = None
        self.db = None

### Test — `Linear` gradient check

In [111]:
# Uses numeric_gradient + _check from Step 0.
# Trick: wrap the layer in a SCALAR loss  L = sum(Y * dY)  (so dL/dY = dY),
# then numeric_gradient of L w.r.t. each of W, b, X must match backward().
def gradient_check_linear(n_in=4, n_out=3, batch=5, seed=1):
    rng = np.random.default_rng(seed)
    layer = Linear(n_in, n_out)
    X  = rng.standard_normal((batch, n_in))
    dY = rng.standard_normal((batch, n_out))     # random upstream (not all-ones)

    W0, b0 = layer.W.copy(), layer.b.copy()

    # analytic gradients from the layer's own backward
    layer.forward(X)
    dX = layer.backward(dY)
    dW_a, db_a = layer.dW.copy(), layer.db.copy()

    # numeric dW: vary W, hold X/b fixed
    def loss_W(Wf):
        layer.W = Wf.reshape(W0.shape)
        return np.sum(layer.forward(X) * dY)
    dW_n = numeric_gradient(loss_W, W0.copy()); layer.W = W0.copy()

    # numeric db: vary b
    def loss_b(bf):
        layer.b = bf.reshape(b0.shape)
        return np.sum(layer.forward(X) * dY)
    db_n = numeric_gradient(loss_b, b0.copy()); layer.b = b0.copy()

    # numeric dX: vary X, params at originals
    def loss_X(Xf):
        return np.sum(layer.forward(Xf.reshape(X.shape)) * dY)
    dX_n = numeric_gradient(loss_X, X.copy())

    _check("dLinear/dW", dW_a, dW_n)
    _check("dLinear/db", db_a, db_n)
    _check("dLinear/dX", dX,   dX_n)

gradient_check_linear()

[PASS] dLinear/dW
[PASS] dLinear/db
[PASS] dLinear/dX


## Step 2 — Cross-Entropy Loss

In [112]:
def cross_entropy(
    rawClassScores: np.ndarray,
    correctClassIndices: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Softmax cross-entropy for a batch of classification examples.

    Args:
        rawClassScores      : (numberOfExamplesInBatch, numberOfClasses)
                              raw scores (logits), one row per example
        correctClassIndices : (numberOfExamplesInBatch,)
                              correct class index for each example

    Returns:
        meanLoss          : float        mean cross-entropy over the batch
        gradientWrtScores : np.ndarray   (numberOfExamplesInBatch, numberOfClasses)
                            dL/dScores = (softmax - onehot) / numberOfExamplesInBatch
    """
    numberOfExamplesInBatch: int = rawClassScores.shape[0]
    exampleRows: np.ndarray = np.arange(numberOfExamplesInBatch)   # [0, 1, ..., batch-1], to index each row

    # --- stable softmax, per row (subtract each row's max -> no overflow) ---
    stabilizedScores: np.ndarray = rawClassScores - rawClassScores.max(axis=1, keepdims=True)
    exponentiatedScores: np.ndarray = np.exp(stabilizedScores)
    classProbabilities: np.ndarray = exponentiatedScores / exponentiatedScores.sum(axis=1, keepdims=True)

    # --- loss: -log(prob of the correct class), averaged over the batch ---
    probabilityOfCorrectClass: np.ndarray = classProbabilities[exampleRows, correctClassIndices]
    negativeLogProbOfCorrectClass: np.ndarray = -np.log(probabilityOfCorrectClass)   # (batch,)
    meanLoss: float = float(negativeLogProbOfCorrectClass.mean())

    # --- gradient: the clean combined form  softmax - onehot  ---
    #   start from probabilities, subtract 1 at each row's correct class, average over batch
    gradientWrtScores: np.ndarray = classProbabilities.copy()
    gradientWrtScores[exampleRows, correctClassIndices] -= 1
    gradientWrtScores /= numberOfExamplesInBatch

    return meanLoss, gradientWrtScores

### Test — `cross_entropy` gradient check

In [113]:
# The combined gradient softmax - onehot must match finite differences of the loss.
def gradient_check_cross_entropy(numberOfExamplesInBatch=4, numberOfClasses=5, seed=1):
    randomGenerator = np.random.default_rng(seed)
    randomScores = randomGenerator.standard_normal((numberOfExamplesInBatch, numberOfClasses))
    correctClassIndices = randomGenerator.integers(0, numberOfClasses, size=numberOfExamplesInBatch)

    analyticLoss, analyticGradient = cross_entropy(randomScores, correctClassIndices)

    # numeric gradient: loss as a scalar function of the scores
    def lossAsFunctionOfScores(flattenedScores):
        lossValue, _ = cross_entropy(flattenedScores.reshape(randomScores.shape), correctClassIndices)
        return lossValue
    numericGradient = numeric_gradient(lossAsFunctionOfScores, randomScores.copy())

    _check("dCE/dScores", analyticGradient, numericGradient)

gradient_check_cross_entropy()

[PASS] dCE/dScores


## Step 2 — Adam Optimizer

In [114]:
class Adam:
    """Adam optimizer: updates every parameter of the given layers in place.

    Holds per-parameter state (first/second moment estimates) and shared state
    (timestep, learning rate, betas, epsilon). Each parameter's update is fully
    independent -- the optimizer just loops over all of them.

    Design:
        - references the layers, so it can reach every parameter
        - reads gradients FRESH each step via layer.parameters()
          (backward rebinds dW/db to new arrays every call)
        - updates values IN PLACE (value -= ...), so the layer's W / b actually change
    """

    def __init__(
        self,
        layers: list,
        lr: float = 1e-3,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ) -> None:
        self.layers: list = layers
        self.lr: float = lr
        self.beta1: float = beta1
        self.beta2: float = beta2
        self.eps: float = eps
        self.timeStep: int = 0

        # one moment buffer per parameter, in the stable order parameters() yields.
        # sized from the VALUES (grads may still be None before the first backward).
        self.firstMomentEstimates: list[np.ndarray] = []
        self.secondMomentEstimates: list[np.ndarray] = []
        for layer in self.layers:
            for (parameterValue, _parameterGradient) in layer.parameters():
                self.firstMomentEstimates.append(np.zeros_like(parameterValue))
                self.secondMomentEstimates.append(np.zeros_like(parameterValue))

    def step(self) -> None:
        """Apply one Adam update to every parameter, using the current gradients."""
        self.timeStep += 1
        biasCorrectionFirst: float = 1.0 - self.beta1 ** self.timeStep
        biasCorrectionSecond: float = 1.0 - self.beta2 ** self.timeStep

        parameterIndex: int = 0
        for layer in self.layers:
            for (parameterValue, parameterGradient) in layer.parameters():   # fresh grads
                firstMoment: np.ndarray = self.firstMomentEstimates[parameterIndex]
                secondMoment: np.ndarray = self.secondMomentEstimates[parameterIndex]

                # update biased moment estimates (in place, so buffer identity is kept)
                firstMoment *= self.beta1
                firstMoment += (1.0 - self.beta1) * parameterGradient
                secondMoment *= self.beta2
                secondMoment += (1.0 - self.beta2) * (parameterGradient ** 2)

                # bias-corrected estimates (early steps would otherwise be too small)
                correctedFirstMoment: np.ndarray = firstMoment / biasCorrectionFirst
                correctedSecondMoment: np.ndarray = secondMoment / biasCorrectionSecond

                # in-place parameter update -> mutates the layer's W / b
                parameterValue -= self.lr * correctedFirstMoment / (np.sqrt(correctedSecondMoment) + self.eps)

                parameterIndex += 1

    def zero_grad(self) -> None:
        """Reset every layer's gradient buffers after a step."""
        for layer in self.layers:
            layer.zero_grad()

### Test — `Adam` reduces loss on a learnable task

In [115]:
# Full training loop on a linearly-separable task: a single Linear + cross_entropy,
# optimized by Adam. Labels come from a linear rule, so the model can actually fit them.
def test_adam_reduces_loss(seed=0):
    randomGenerator = np.random.default_rng(seed)
    numberOfExamples, numberOfFeatures, numberOfClasses = 64, 5, 3

    inputs = randomGenerator.standard_normal((numberOfExamples, numberOfFeatures))
    trueWeights = randomGenerator.standard_normal((numberOfFeatures, numberOfClasses))
    correctClassIndices = np.argmax(inputs @ trueWeights, axis=1)   # learnable labels

    layer = Linear(numberOfFeatures, numberOfClasses)
    optimizer = Adam([layer], lr=0.1)

    weightsBefore = layer.W.copy()
    firstLoss = None
    lastLoss = None
    for stepIndex in range(300):
        rawClassScores = layer.forward(inputs)
        loss, gradientWrtScores = cross_entropy(rawClassScores, correctClassIndices)
        layer.backward(gradientWrtScores)
        optimizer.step()
        optimizer.zero_grad()
        if stepIndex == 0:
            firstLoss = loss
        lastLoss = loss

    predictions = np.argmax(layer.forward(inputs), axis=1)
    accuracy = float((predictions == correctClassIndices).mean())

    _check("Adam drives loss down", lastLoss < firstLoss * 0.5, True)
    _check("Adam updated the weights in place", np.any(layer.W != weightsBefore), True)
    _check("fits the learnable task (acc > 0.9)", accuracy > 0.9, True)
    print(f"   loss: {firstLoss:.3f} -> {lastLoss:.3f}   accuracy: {accuracy:.2f}")

test_adam_reduces_loss()

[PASS] Adam drives loss down
[PASS] Adam updated the weights in place
[PASS] fits the learnable task (acc > 0.9)
   loss: 2.672 -> 0.065   accuracy: 0.98


## CharTokenizer — text ↔ tokens

Character-level tokenizer: converts text to integer token ids and back. Not a Layer — it has
no parameters and the optimizer never touches it. Vocabulary = special tokens
(`<pad>`, `<bos>`, `<eos>`) + every unique character in the training text.

In [116]:
class CharTokenizer:
    """Character-level tokenizer: text <-> list of integer token ids.

    Vocabulary = special tokens + every unique character in the training text.
    Not a Layer: no parameters, never seen by the optimizer.
    """

    PAD_TOKEN: str = "<pad>"
    BOS_TOKEN: str = "<bos>"   # beginning of sequence
    EOS_TOKEN: str = "<eos>"   # end of sequence

    def __init__(self, trainingText: str) -> None:
        specialTokens: list[str] = [self.PAD_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]
        uniqueCharacters: list[str] = sorted(set(trainingText))
        # id -> token (list index is the id); token -> id (inverse map)
        self.idToToken: list[str] = specialTokens + uniqueCharacters
        self.tokenToId: dict[str, int] = {token: tokenId for tokenId, token in enumerate(self.idToToken)}

    @property
    def vocabSize(self) -> int:
        return len(self.idToToken)

    @property
    def padId(self) -> int:
        return self.tokenToId[self.PAD_TOKEN]

    @property
    def bosId(self) -> int:
        return self.tokenToId[self.BOS_TOKEN]

    @property
    def eosId(self) -> int:
        return self.tokenToId[self.EOS_TOKEN]

    def encode(self, text: str, addSpecials: bool = True) -> list[int]:
        """text -> token ids. If addSpecials, wrap as <bos> ... <eos>."""
        tokenIds: list[int] = [self.tokenToId[character] for character in text]
        if addSpecials:
            tokenIds = [self.bosId] + tokenIds + [self.eosId]
        return tokenIds

    def decode(self, tokenIds: list[int], skipSpecials: bool = True) -> str:
        """token ids -> text. If skipSpecials, drop <pad>/<bos>/<eos>."""
        specialTokenIds: set[int] = {self.padId, self.bosId, self.eosId}
        return "".join(
            self.idToToken[tokenId]
            for tokenId in tokenIds
            if not (skipSpecials and tokenId in specialTokenIds)
        )

### Test — `CharTokenizer` round-trip

In [117]:
# _check uses np.allclose (numeric); add a plain-equality check for strings/ints/lists.
def _check_eq(name, got, want):
    ok = (got == want)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", repr(got))
        print("   want:", repr(want))

def test_char_tokenizer():
    tokenizer = CharTokenizer("a dog runs")

    # vocab = 3 special tokens + unique characters of the training text
    uniqueCharacterCount = len(set("a dog runs"))
    _check_eq("vocab size", tokenizer.vocabSize, 3 + uniqueCharacterCount)

    # round-trip: decode(encode(text)) recovers the text (specials skipped on decode)
    for text in ["a dog", "runs", "a", "dog runs"]:
        _check_eq(f"round-trip {text!r}", tokenizer.decode(tokenizer.encode(text)), text)

    # encode wraps with <bos> ... <eos>
    tokenIds = tokenizer.encode("a")
    _check_eq("starts with <bos>", tokenIds[0], tokenizer.bosId)
    _check_eq("ends with <eos>", tokenIds[-1], tokenizer.eosId)

    # decode can keep the special tokens when asked
    _check_eq("decode keeps specials", tokenizer.decode(tokenIds, skipSpecials=False), "<bos>a<eos>")

    # addSpecials=False encodes just the characters
    _check_eq("no specials when off", tokenizer.encode("a", addSpecials=False), [tokenizer.tokenToId["a"]])

test_char_tokenizer()

[PASS] vocab size
[PASS] round-trip 'a dog'
[PASS] round-trip 'runs'
[PASS] round-trip 'a'
[PASS] round-trip 'dog runs'
[PASS] starts with <bos>
[PASS] ends with <eos>
[PASS] decode keeps specials
[PASS] no specials when off


## Embedding — token id → learned vector

A Layer with **one** parameter: the table `(vocabSize, d)`, one row per token.
- **forward** is a row lookup: `table[ids]` (no computation, just indexing).
- **backward** is a **scatter-add**: each used row receives the gradient of every position
  that used it (duplicate ids accumulate). It returns **`None`** — integer ids aren't
  differentiable, and Embedding is always the first layer.

Same `parameters()` contract as `Linear`, so `Adam` updates its table with no special-casing.

In [118]:
class Embedding:
    """Embedding layer: maps integer token ids to learned vectors (a row lookup).

    One trainable parameter: table (vocabSize, embeddingDim), one row per token.
        forward(ids)  -> table[ids]   (rows selected; caches ids)
        backward(dOut) -> None        (scatter-add into dTable; duplicate ids accumulate)

    Returns no input gradient — integer ids aren't differentiable, and Embedding is the
    first layer. Same parameters() contract as Linear, so the optimizer treats it the same.
    """

    def __init__(self, vocabSize: int, embeddingDim: int, seed: int = 0) -> None:
        randomGenerator = np.random.default_rng(seed)
        self.table: np.ndarray = randomGenerator.standard_normal((vocabSize, embeddingDim))
        # cache + gradient buffer (filled during forward/backward)
        self.tokenIds: np.ndarray | None = None
        self.dTable: np.ndarray | None = None

    def forward(self, tokenIds: np.ndarray) -> np.ndarray:
        """tokenIds: integer array of shape S -> embeddings of shape S + (embeddingDim,)."""
        self.tokenIds = tokenIds          # cache ids: backward needs to know which rows were used
        return self.table[tokenIds]       # fancy indexing = row lookup

    def backward(self, dOut: np.ndarray) -> None:
        """dOut: dL/d(looked-up rows), same shape as the forward output.

        Scatter-add each position's gradient into the row of its id. np.add.at accumulates
        on repeated ids (plain dTable[ids] = dOut would overwrite duplicates -> wrong).
        Returns None: there is no gradient to pass back to integer ids.
        """
        self.dTable = np.zeros_like(self.table)
        np.add.at(self.dTable, self.tokenIds, dOut)
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(table, dTable)]."""
        return [(self.table, self.dTable)]

    def zero_grad(self) -> None:
        """Reset the gradient buffer (called after each optimizer step)."""
        self.dTable = None

### Test — `Embedding` gradient check (scatter-add)

In [119]:
# Scatter-add backward must match finite differences of the table.
# Use a REPEATED id so the accumulation path is exercised.
def gradient_check_embedding(vocabSize=6, embeddingDim=4, seed=1):
    randomGenerator = np.random.default_rng(seed)
    embedding = Embedding(vocabSize, embeddingDim)

    tokenIds = np.array([2, 0, 4, 2, 2])   # id 2 appears 3x -> tests scatter-add accumulation
    upstreamGradient = randomGenerator.standard_normal((len(tokenIds), embeddingDim))

    # forward shape sanity
    embeddedRows = embedding.forward(tokenIds)
    _check_eq("forward shape", embeddedRows.shape, (len(tokenIds), embeddingDim))

    # analytic gradient from scatter-add backward
    returnValue = embedding.backward(upstreamGradient)
    analyticGradient = embedding.dTable.copy()
    _check_eq("backward returns None", returnValue, None)

    # accumulation: the repeated id's row = sum of the gradients at its occurrences (0, 3, 4)
    _check("scatter-add accumulates", analyticGradient[2], upstreamGradient[[0, 3, 4]].sum(axis=0))

    # numeric gradient: L = sum(table[ids] * upstreamGradient) as a function of the table
    originalTable = embedding.table.copy()
    def lossAsFunctionOfTable(flattenedTable):
        embedding.table = flattenedTable.reshape(originalTable.shape)
        return np.sum(embedding.forward(tokenIds) * upstreamGradient)
    numericGradient = numeric_gradient(lossAsFunctionOfTable, originalTable.copy())
    embedding.table = originalTable

    _check("dEmbedding/dTable", analyticGradient, numericGradient)

gradient_check_embedding()

[PASS] forward shape
[PASS] backward returns None
[PASS] scatter-add accumulates
[PASS] dEmbedding/dTable


## Bigram — next token from the current token

The first assembled model, and the integration test for everything so far. It predicts the
next token from **only the current token**: `Embedding(vocab, d) → Linear(d, vocab) → logits`.

- **`forward`** (both phases) — embed the current token, project to next-token logits.
- **`backward`** (training) — push the loss gradient back through Linear, then Embedding.
- **`generate`** (working) — forward only: sample a token, feed it back, until `<eos>`.

`parameters()` / `zero_grad()` just delegate to the two sub-layers, so `Adam([model])` updates
both with no special-casing.

In [120]:
class Bigram:
    """Bigram language model: predicts the next token from ONLY the current token.

    Composition:  Embedding(vocab, d) -> Linear(d, vocab) -> next-token logits.
    forward + backward are used in training; forward alone drives generation.
    """

    def __init__(self, vocabSize: int, embeddingDim: int, seed: int = 0) -> None:
        self.vocabSize: int = vocabSize
        self.embedding: Embedding = Embedding(vocabSize, embeddingDim, seed=seed)
        self.projection: Linear = Linear(embeddingDim, vocabSize, seed=seed + 1)

    def forward(self, currentTokenIds: np.ndarray) -> np.ndarray:
        """currentTokenIds: (batch,) -> logits (batch, vocabSize)."""
        embedded: np.ndarray = self.embedding.forward(currentTokenIds)   # (batch, d)
        logits: np.ndarray = self.projection.forward(embedded)          # (batch, vocab)
        return logits

    def backward(self, gradientWrtLogits: np.ndarray) -> None:
        """Backprop the loss gradient through projection, then embedding."""
        gradientWrtEmbedded: np.ndarray = self.projection.backward(gradientWrtLogits)  # (batch, d)
        self.embedding.backward(gradientWrtEmbedded)   # scatter-add into the table; returns None
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """All (value, gradient) pairs, so one optimizer can update the whole model."""
        return self.embedding.parameters() + self.projection.parameters()

    def zero_grad(self) -> None:
        self.embedding.zero_grad()
        self.projection.zero_grad()

    def generate(
        self,
        tokenizer: CharTokenizer,
        maxNewTokens: int = 100,
        temperature: float = 1.0,
        seed: int = 0,
    ) -> str:
        """Autoregressive generation: start at <bos>, sample tokens until <eos> or the cap."""
        randomGenerator = np.random.default_rng(seed)
        generatedIds: list[int] = [tokenizer.bosId]

        for _ in range(maxNewTokens):
            currentTokenId: np.ndarray = np.array([generatedIds[-1]])       # (1,)
            logits: np.ndarray = self.forward(currentTokenId)[0]           # (vocab,)
            probabilities: np.ndarray = stable_softmax(logits / temperature)
            probabilities = probabilities / probabilities.sum()            # guard against rounding
            nextTokenId: int = int(randomGenerator.choice(self.vocabSize, p=probabilities))
            generatedIds.append(nextTokenId)
            if nextTokenId == tokenizer.eosId:
                break

        return tokenizer.decode(generatedIds)

### Train + generate — first text from a from-scratch LM

Train on a small structured corpus with the full stack (`Embedding → Linear → cross_entropy →
Adam`) on shift-by-one pairs, then generate. Success = loss beats the uniform baseline
`ln(vocab)` and the output forms letter patterns.

In [121]:
def train_bigram(
    model: Bigram,
    tokenizer: CharTokenizer,
    text: str,
    steps: int = 500,
    lr: float = 0.1,
) -> list[float]:
    """Full-batch training on shift-by-one pairs. Returns the loss at each step."""
    tokenIds: np.ndarray = np.array(tokenizer.encode(text))   # includes <bos> ... <eos>
    currentTokenIds: np.ndarray = tokenIds[:-1]               # inputs
    nextTokenIds: np.ndarray = tokenIds[1:]                   # targets (shifted by one)

    optimizer = Adam([model], lr=lr)
    lossHistory: list[float] = []
    for _ in range(steps):
        logits = model.forward(currentTokenIds)
        loss, gradientWrtLogits = cross_entropy(logits, nextTokenIds)
        model.backward(gradientWrtLogits)
        optimizer.step()
        optimizer.zero_grad()
        lossHistory.append(loss)
    return lossHistory


def test_bigram():
    corpusText = "the quick brown fox jumps over the lazy dog. " * 30
    tokenizer = CharTokenizer(corpusText)
    model = Bigram(tokenizer.vocabSize, embeddingDim=32)

    lossHistory = train_bigram(model, tokenizer, corpusText, steps=500, lr=0.1)

    uniformBaseline = float(np.log(tokenizer.vocabSize))   # loss of random guessing
    _check("loss decreased", lossHistory[-1] < lossHistory[0], True)
    _check("beats uniform baseline ln(vocab)", lossHistory[-1] < uniformBaseline, True)
    print(f"   loss: {lossHistory[0]:.3f} -> {lossHistory[-1]:.3f}   (uniform baseline {uniformBaseline:.3f})")

    sample = model.generate(tokenizer, maxNewTokens=60, temperature=0.8, seed=1)
    print("   sample:", repr(sample))

test_bigram()

[PASS] loss decreased
[PASS] beats uniform baseline ln(vocab)
   loss: 12.641 -> 0.639   (uniform baseline 3.434)
   sample: 'ther juick ove lazy quick overog. ox lazy ove lazy the brox '
